# Reading a video with g2ltk

In this notebook, we open one of the reference test videos, print its main features (image size, frame rate, duration) and show some of its frames.

The test videos are:
- `test_gcv.gcv/` : GevCapture video (meta + stamps + raw video)
- `test_mp4.mp4` : lossy compressed video (H.264 codec)
- `test_mkv.mkv` : losslessly compressed video (FFV1 codec)

In [ ]:
%matplotlib widget

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

from g2ltk import videoreading

## Selecting the video

There are two ways of selecting a video to use in g2ltk
* Simply provide an `acquisition_path` (path to the video without extension)
* Or define a folder, termed `dataset`, containing the video, termed `acquisition`


In [ ]:
### Simple method: define acquisition_path
test_data_dir = os.path.join('..', 'test_data')
acquisition_path = os.path.join(test_data_dir, 'test_gcv') # 'gcv', 'mp4' or 'mkv'
print(f'acquisition_path: {acquisition_path}')

if not videoreading.is_this_a_video(acquisition_path):
    raise FileNotFoundError(f'No video found at {acquisition_path}')

In [ ]:
### 'g2ltk' method: define a dataset containing acquisitions

# The *root path* indicates where we are going to find the datasets (directories containing videos)
videoreading.set_default_root_path('..')

# A *dataset* is the name of a directory containing videos (*acquisitions*) of interest
dataset = 'test_data'
videoreading.describe_dataset(dataset=dataset, makeitshort=False)

# An *acquisition*  is the name of video file (without extension)
acquisition = 'test_gcv' # 'gcv', 'mp4' or 'mkv'
videoreading.describe_acquisition(acquisition=acquisition, dataset=dataset)

acquisition_path = videoreading.generate_acquisition_path(acquisition=acquisition, dataset=dataset)
print(f'acquisition_path: {acquisition_path}')

## Video features

In [ ]:
# Format as detected by g2ltk
type_checkers = {'gcv': videoreading.is_this_a_gcv, 'mkv': videoreading.is_this_a_mkv, 'mp4': videoreading.is_this_a_mp4}
detected_type = next((name for name, is_this_a in type_checkers.items() if is_this_a(acquisition_path)), 'other')

n_frames, height, width = videoreading.get_geometry(acquisition_path)
frame_rate = videoreading.get_acquisition_frequency(acquisition_path, unit='Hz')
duration = videoreading.get_acquisition_duration(acquisition_path, framenumbers=None, unit='s')

print(f'Video          : {acquisition_path} (detected type: {detected_type})')
print(f'Image size     : {width} x {height} px (width x height)')
print(f'Frames         : {n_frames}')
print(f'Frame rate     : {frame_rate:.3f} Hz')
print(f'Duration       : {duration:.3f} s (from first to last frame)')

## Loading the frames

`get_frames` returns an array of shape `(n_frames, height, width)`.
The times of the frames, in seconds, are given by `get_t_s`.

In [ ]:
frames = videoreading.get_frames(acquisition_path)
t_s = videoreading.get_t_s(acquisition_path)

print(f'Frames array   : shape {frames.shape}, dtype {frames.dtype}, values in [{frames.min()}, {frames.max()}]')

## First, middle and last frames

In [ ]:
framenumbers_to_show = [0, n_frames // 2, n_frames - 1]

fig, axes = plt.subplots(1, len(framenumbers_to_show), figsize=(9, 3.4), sharex=True, sharey=True, constrained_layout=True)
for ax, fn in zip(axes, framenumbers_to_show):
    ax.imshow(frames[fn], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'frame {fn} (t = {t_s[fn]:.3f} s)')
    ax.set_xlabel('x [px]')
axes[0].set_ylabel('z [px]')
plt.show()

## Browsing the video

Use the slider to go through the frames.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5.5))
fig.subplots_adjust(bottom=0.18)

image = ax.imshow(frames[0], cmap='gray', vmin=0, vmax=255)
ax.set_xlabel('x [px]')
ax.set_ylabel('z [px]')
ax.set_title(f'frame 0 (t = {t_s[0]:.3f} s)')

slider_ax = fig.add_axes([0.15, 0.04, 0.7, 0.04])
frame_slider = Slider(slider_ax, 'frame', 0, n_frames - 1, valinit=0, valstep=1)

def show_frame(framenumber):
    fn = int(framenumber)
    image.set_data(frames[fn])
    ax.set_title(f'frame {fn} (t = {t_s[fn]:.3f} s)')
    fig.canvas.draw_idle()

frame_slider.on_changed(show_frame)
plt.show()